# Waste BAU uplift — subir 4.A+4.D de ~10.5 Mt a ~28 Mt (2050)

**Objetivo.** El BAU actual produce **10.5 MtCO2e** en Waste (4.A Solid Waste 7.4 + 4.D Wastewater 3.1) en 2050. La SNBC Reference (Figura 36, p.63) muestra **~28-30 Mt** en 2050 (~25 sólidos + ~3 aguas residuales). Gap: **+17 Mt al 2050**, casi todo en sólidos.

**Diagnóstico.** En `sisepuede_raw_input_morocco_fuels.csv`:

| Driver | Valor actual | Problema |
|---|---|---|
| `elasticity_waso_msw_to_gdppc_*` (11 streams) | **0.01** constante | Cero efecto del ingreso en generación de MSW |
| `factor_waso_waste_per_capita_scalar_*` | 1.0 constante | OK, sin step-changes |
| `physparam_wali_daily_kg_bod_per_capita` | 0.045 (IPCC default) | OK |
| Treatment paths (frac_wali_ww_*) | Constantes | OK |

El problema es uno: con elasticidad ~0, la generación de residuos sólo crece por población (~1.3x al 2050). SNBC asume urbanización + consumo aumentan MSW per cápita fuertemente con el ingreso. Para ir de 5.1 → 25 Mt en sólidos (~5x) con población 1.3x, hace falta elasticidad efectiva ~1.0.

**Estrategia.** Subir `elasticity_waso_msw_to_gdppc_*` de **0.01 → 0.80** desde tp=8 para los 11 streams que escalan con consumo:
- food, paper, plastic, textiles, nappies, glass, metal, rubber_leather, wood, yard, other
- **No tocar** `chemical_industrial` ni `sludge` (esos dependen de industria, no de ingreso doméstico)

**Wastewater (4.D).** En tu BAU ya pasa de 1.7 → 3.1 Mt (1.8x), consistente con crecimiento poblacional. SNBC muestra ~3 Mt en 2050. **No tocar wastewater.**

**Salida.** `sisepuede_raw_input_morocco_fuels_waste_bau.csv` en `input_data/`.

In [11]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path('/Users/fabianfuentes/git/ssp_morocco')
INPUT_DIR = REPO / 'ssp_modeling' / 'input_data'

INPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_scoe_inen_ippu.csv'
OUTPUT_FILE = INPUT_DIR / 'sisepuede_raw_input_morocco_fuels_scoe_inen_ippu_waste.csv'

# --- Knobs ---
TP_CALIB_END = 7

# Nueva elasticidad MSW→GDPpc para tp>7 (los 11 streams dependientes del ingreso)
NEW_ELASTICITY = 0.60

# Streams a tocar (excluir chemical_industrial y sludge que dependen de industria)
INCOME_DRIVEN_STREAMS = [
    'food', 'glass', 'metal', 'nappies', 'other', 'paper',
    'plastic', 'rubber_leather', 'textiles', 'wood', 'yard',
]

pd.options.display.float_format = '{:.4f}'.format
df = pd.read_csv(INPUT_FILE)
print(f'Loaded {INPUT_FILE.name}: {df.shape}  tp=[{df.time_period.min()}, {df.time_period.max()}]')

Loaded sisepuede_raw_input_morocco_fuels_scoe_inen_ippu.csv: (56, 2442)  tp=[0, 55]


## 1 · Auditoría — elasticidades actuales

In [12]:
ELAS_COLS = [f'elasticity_waso_msw_to_gdppc_{s}' for s in INCOME_DRIVEN_STREAMS]
missing = [c for c in ELAS_COLS if c not in df.columns]
assert not missing, f'Faltan columnas: {missing}'

snapshot_tps = [0, 7, 8, 15, 25, 35, 45, 55]
df.loc[df.time_period.isin(snapshot_tps), ['time_period', 'year'] + ELAS_COLS[:5]].set_index('time_period')

,year,elasticity_waso_msw_to_gdppc_food,elasticity_waso_msw_to_gdppc_glass,elasticity_waso_msw_to_gdppc_metal,elasticity_waso_msw_to_gdppc_nappies,elasticity_waso_msw_to_gdppc_other
time_period,,,,,,
0,2015,0.0100,0.0100,0.0100,0.0100,0.0100
7,2022,0.0100,0.0100,0.0100,0.0100,0.0100
8,2023,0.0100,0.0100,0.0100,0.0100,0.0100
15,2030,0.0100,0.0100,0.0100,0.0100,0.0100
25,2040,0.0100,0.0100,0.0100,0.0100,0.0100
35,2050,0.0100,0.0100,0.0100,0.0100,0.0100
45,2060,0.0100,0.0100,0.0100,0.0100,0.0100
55,2070,0.0100,0.0100,0.0100,0.0100,0.0100


## 2 · Aplicar cambios

Step a `NEW_ELASTICITY` desde tp=8 (tp 0-7 quedan en 0.01 para no romper la calibración histórica de NIR).

In [13]:
df_new = df.copy()
forward = df_new['time_period'] > TP_CALIB_END
for col in ELAS_COLS:
    df_new.loc[forward, col] = NEW_ELASTICITY

print('=== After (3 streams de ejemplo) ===')
df_new.loc[df_new.time_period.isin(snapshot_tps), ['time_period', 'year'] + ELAS_COLS[:3]].set_index('time_period')

=== After (3 streams de ejemplo) ===


,year,elasticity_waso_msw_to_gdppc_food,elasticity_waso_msw_to_gdppc_glass,elasticity_waso_msw_to_gdppc_metal
time_period,,,,
0,2015,0.0100,0.0100,0.0100
7,2022,0.0100,0.0100,0.0100
8,2023,0.6000,0.6000,0.6000
15,2030,0.6000,0.6000,0.6000
25,2040,0.6000,0.6000,0.6000
35,2050,0.6000,0.6000,0.6000
45,2060,0.6000,0.6000,0.6000
55,2070,0.6000,0.6000,0.6000


## 3 · Verificación — calibración histórica intacta

In [14]:
diff = (df_new[ELAS_COLS] - df[ELAS_COLS]).abs()
calib_max = diff.loc[df['time_period'] <= TP_CALIB_END].max().max()
print(f'Max diff en tp=0..7: {calib_max:.2e}')
assert calib_max < 1e-12, 'Calibración tocada — abortar.'
print(f'OK: calibración intacta. {len(ELAS_COLS)} columnas modificadas para tp>{TP_CALIB_END}.')
print(f'Cambio: 0.01 → {NEW_ELASTICITY:.2f} en cada una.')

Max diff en tp=0..7: 0.00e+00
OK: calibración intacta. 11 columnas modificadas para tp>7.
Cambio: 0.01 → 0.60 en cada una.


## 4 · Guardar

In [15]:
df_new.to_csv(OUTPUT_FILE, index=False)
print(f'Wrote {OUTPUT_FILE}')
print(f'Shape: {df_new.shape}')

Wrote /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/input_data/sisepuede_raw_input_morocco_fuels_scoe_inen_ippu_waste.csv
Shape: (56, 2442)


## 5 · Iteración

Correr el modelo apuntando a `sisepuede_raw_input_morocco_fuels_waste_bau.csv`. **OJO con el lag del modelo de descomposición de primer orden de relleno sanitario:** las emisiones de CH4 del 2050 dependen de los residuos generados en 2030-2045 (tiempo de vida media del relleno). Por eso aunque subas la elasticidad, la curva BAU al 2050 reflejará el integral de generación 2018-2050.

Targets de validación:
- **4.A Solid Waste 2050**: 22-26 Mt (SNBC ~25 Mt).
- **4.D Wastewater 2050**: 3-3.5 Mt (sin tocar, debe quedar igual ~3.1).
- **Total Waste 2050**: 25-29 Mt (SNBC ~28 Mt).

Si te quedas:
- **<22 Mt**: subir `NEW_ELASTICITY` a 1.00 o 1.10.
- **>30 Mt**: bajar a 0.60.

**Si la curva sigue muy plana en 2030 y dispara solo después de 2040**, es por el lag de descomposición. En ese caso considera además mover `factor_waso_waste_per_capita_scalar_food` a 1.15-1.20 para meter step-change de generación temprana (mayor consumo per cápita post-2025).

**Por qué no toqué wastewater.** En tu BAU 4.D ya crece 1.7→3.1 Mt (~1.8x), alineado con crecimiento de población urbana + cobertura de alcantarillado. SNBC muestra ~3 Mt en 2050 (idéntico). Si quisieras subirlo, los drivers serían `physparam_wali_daily_kg_bod_per_capita` (0.045 → 0.055, dieta más proteínica) o aumentar fracciones de tratamiento anaeróbico vs aeróbico.